# **Synteny Analysis – Flankophile**

## Tool Information

- **Tool:** Flankophile 
- **Input:** Annotated genomes (GFF + FASTA)  
- **Organism:** *Acinetobacter baumannii*  
- **Analysis type:** Flanking region extraction and gene context analysis  

Flankophile is used to extract genomic regions surrounding genes of interest, particularly antimicrobial resistance (AMR) genes. It enables analysis of upstream and downstream regions to identify mobile genetic elements (MGEs), insertion sequences (IS), and gene neighbourhood structures.

This analysis focuses on extracting flanking regions of key AMR genes to study their genomic context and potential mobility.


## **Installation and Setup**

### 1) Clone repository
This step downloads the Flankophile source code from GitHub into your working directory.

In [18]:
%%bash
cd /home/nidhi/tools
git clone https://bitbucket.org/genomicepidemiology/flankophile.git
cd flankophile
ls


Cloning into 'flankophile'...


README.md
Snakefile
bin
bitbucket-pipelines.yml
config.yaml
environment.yaml
example_output
input
quick_start.md
version_log.txt


### 2) Load Conda and create an environment for Flankophile
Loads conda into the shell so environments can be created and activated then creates a clean conda environment for running Flankophile.

In [ ]:
%%bash
source ~/.bashrc

conda create -y \
  -p /home/nidhi/.conda/envs/flankophile_aba \
  python=3.9

### 3) Install Dependencies (Recommended via Mamba)
Mamba is used for faster and more reliable package installation. The required tools include:

a) snakemake (Mandatory): The central workflow engine required to automate and run the entire pipeline's rules and logic.

b) samtools (Optional): A fundamental utility used for indexing reference files and processing alignment data within the workflow.

c) bedtools (Optional): A critical suite used specifically for extracting target sequences (via getfasta) and masking flanking regions (via maskfasta).

In [ ]:
%%bash
mamba install -c conda-forge -c bioconda snakemake samtools bedtools -y

### 4) Verify Installation
Checks that key tools are installed correctly and checks their current versions

In [23]:
%%bash

snakemake --version | head -n 1
samtools --version 2>/dev/null | head -n 1
bedtools --version

7.32.4
samtools 1.23.1
bedtools v2.31.1


### 5) Dry run Flankophile
This step performs a dry run of the Flankophile pipeline using Snakemake to validate the workflow without executing any jobs. No actual analysis is performed during this step.

In [27]:
%%bash

source /home/anaconda/miniconda3/etc/profile.d/conda.sh
conda activate flankophile_aba

cd /home/nidhi/tools/flankophile

snakemake --snakefile Snakefile -n --cores 1

## **Input Files for Flankophile**

Flankophile requires two primary input components:

1. A **reference database (FASTA)**
2. An **input list file (TSV) describing sample genomes**

These file paths must be specified in the `config.yaml` file.

### 1. Reference Database (FASTA)

The reference database contains the **target sequences (e.g., genes of interest)** for which flanking regions will be analyzed.

**Requirements:**

* Must be a **DNA multi-FASTA file**
* Each sequence must have a **unique header and unique sequence**
* Can contain **one or many sequences**
* Can include:

  * AMR genes (e.g., ResFinder database)
  * Any custom DNA sequences of interest

Flankophile will cluster reference sequences based on similarity and analyze each cluster separately.

### 2. Input List File (TSV)

The input list is a **tab-separated file** describing all genome assemblies to be analyzed.

**Columns:**

1. **Assembly Name**: A unique nickname for each input fasta

2. **Path to FASTA**: 

   * Full path to the genome file
   * Must be a **multi-FASTA file**
   * One file per sample

3. **Metadata (optional)**

   * Additional grouping information (e.g., host, location)
   * Used for coloring and annotation in plots

Example:

```
#assembly_name	path	metadata
sample1	/data/genomes/sample1.fas	human
sample2	/data/genomes/sample2.fas	pig
```

Flankophile scans these sequences to identify matches against the reference database and extract flanking regions, the file allows Flankophile to associate each genome with its identity and metadata for downstream visualization.

Below is a simple bash script to make the input file


In [ ]:
%%bash
# create output directory if it does not exist
mkdir -p /data/internship_data/nidhi/aba/new_output/flankophile_output

# create the TSV directly in the output directory
echo -e "assembly_name\tpath\tmetadata" > \
/data/internship_data/nidhi/aba/new_output/flankophile_output/flanko_input_list.tsv

for f in /data/internship_data/nidhi/aba/new_output/nextflow_output/assemblies/*.short.fasta; do
    sample=$(basename "$f" .short.fasta)
    echo -e "${sample}\t${f}\tNA" >> \
    /data/internship_data/nidhi/aba/new_output/flankophile_output/flanko_input_list.tsv
done

## Confirm Environment Setup

In [41]:
%%bash
mkdir -p /home/nidhi/.snakemake/conda

In [43]:
%%bash 
mkdir -p /home/nidhi/snakemake_conda_envs

In [37]:
%%bash
snakemake --cleanup-conda

usage: snakemake [-h] [--dry-run] [--profile PROFILE]
                 [--workflow-profile WORKFLOW_PROFILE] [--cache [RULE ...]]
                 [--snakefile FILE] [--cores [N]] [--jobs [N]]
                 [--local-cores N] [--resources [NAME=INT ...]]
                 [--set-threads RULE=THREADS [RULE=THREADS ...]]
                 [--max-threads MAX_THREADS]
                 [--set-resources RULE:RESOURCE=VALUE [RULE:RESOURCE=VALUE ...]]
                 [--set-scatter NAME=SCATTERITEMS [NAME=SCATTERITEMS ...]]
                 [--set-resource-scopes RESOURCE=[global|local]
                 [RESOURCE=[global|local] ...]]
                 [--default-resources [NAME=INT ...]]
                 [--preemption-default PREEMPTION_DEFAULT]
                 [--preemptible-rules PREEMPTIBLE_RULES [PREEMPTIBLE_RULES ...]]
                 [--config [KEY=VALUE ...]] [--configfile FILE [FILE ...]]
                 [--envvars VARNAME [VARNAME ...]] [--directory DIR] [--touch]
                 

CalledProcessError: Command 'b'snakemake --cleanup-conda\n'' returned non-zero exit status 2.

In [47]:
%%bash

source /home/anaconda/miniconda3/etc/profile.d/conda.sh
conda activate flankophile_aba

cd /home/nidhi/tools/flankophile

snakemake \
  --use-conda \
  --conda-frontend conda \
  --conda-prefix /home/nidhi/clean_snakemake_envs \
  --cores 8

/home/nidhi/.conda/envs/aba_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
Building DAG of jobs...
Creating conda environment environment.yaml...


Error while terminating subprocess (pid=1495416): 


Cancelling snakemake on user request.


In [48]:
%%bash

# Load conda
source /home/anaconda/miniconda3/etc/profile.d/conda.sh

echo "🔹 Removing conda environments..."

# Remove environments if they exist
conda env remove -n flankophile_aba -y || true

# Also remove any stray user env folders
rm -rf /home/nidhi/.conda/envs/flankophile_aba

echo "🔹 Removing Snakemake and temp conda caches..."

# Remove Snakemake cache directories
rm -rf /home/nidhi/.snakemake
rm -rf /home/nidhi/clean_snakemake_envs
rm -rf /home/nidhi/tmp_snakemake_envs
rm -rf /home/nidhi/snakemake_conda_envs

echo "🔹 Removing residual conda package caches (optional but recommended)..."

# Clean conda cache (safe)
conda clean --all -y

echo "✅ Cleanup complete"

🔹 Removing conda environments...

Remove all packages in environment /home/nidhi/.conda/envs/flankophile_aba:


## Package Plan ##

  environment location: /home/nidhi/.conda/envs/flankophile_aba


The following packages will be REMOVED:

  _openmp_mutex-4.5-20_gnu
  bzip2-1.0.8-hda65f42_9
  ca-certificates-2026.2.25-hbd8a1cb_0
  ld_impl_linux-64-2.45.1-default_hbd61a6d_102
  libexpat-2.7.5-hecca717_0
  libffi-3.5.2-h3435931_0
  libgcc-15.2.0-he0feb66_18
  libgcc-ng-15.2.0-h69a702a_18
  libgomp-15.2.0-he0feb66_18
  liblzma-5.8.2-hb03c661_0
  libnsl-2.0.1-hb9d3cd8_1
  libsqlite-3.52.0-h0c1763c_0
  libuuid-2.42-h5347b49_0
  libxcrypt-4.4.36-hd590300_1
  libzlib-1.3.2-h25fd6f3_2
  ncurses-6.5-h2d0b736_3
  openssl-3.6.2-h35e630c_0
  pip-22.1.2-pyhd8ed1ab_0
  python-3.9.23-hc30ae73_0_cpython
  readline-8.3-h853b02a_0
  setuptools-80.9.0-pyhff2d567_0
  tk-8.6.13-noxft_h366c992_103
  tzdata-2025c-hc9c84f9_1
  wheel-0.45.1-pyhd8ed1ab_1
  zstd-1.5.7-hb78ec9c_6



Preparing transaction: done
Ver

## Final Environment Validation

In [31]:
%%bash
rm -rf /home/nidhi/tools/flankophile/.snakemake

## Install Dependencies (Step 1)

In [ ]:
10

## Install Dependencies (Step 2)

In [ ]:
11

## Verify Installation

In [ ]:
12

## Prepare Input Data

In [ ]:
15

## Organize Input Files

In [ ]:
16

## Validate Input Files

In [ ]:
17

## Additional Input Preparation

In [ ]:
18

## Final Input Check

In [ ]:
19

## Run Flankophile (Step 1)

In [ ]:
21

## Run Flankophile (Step 2)

In [ ]:
23

## Verify Flankophile Output

In [ ]:
24

## Parse Output Files

In [ ]:
26

## Extract Relevant Results

In [ ]:
27

## Analyze Results (Step 1)

In [ ]:
30

## Analyze Results (Step 2)

In [ ]:
31

## Additional Filtering

In [ ]:
33

## Intermediate Validation

In [ ]:
36

## Final Data Processing

In [ ]:
38

## Generate Final Outputs

In [ ]:
41

## Final Verification

In [ ]:
44